# Demo: a dip with rotation, in the air

Practises the motion of a dip coat with rotation, with the **empty** gripper, high above `mid_station`,
pointing straight down. Nothing touches anything.

- **Cell 1**: settings, then up into the air. You set: `ROTATION_SPEED_DEG_S` (clockwise, seen from above),
  `DIP_SPEED_MM_S` and `DIP_DEPTH_MM` (going down), `HOLD_S`, `LIFT_SPEED_MM_S` (coming up). It works out how
  far the gripper will turn in all and refuses if that is more than the joint allows (about 2 turns).
- **Cell 2**:
  - a) a 10-degree turn to learn which way is clockwise;
  - b) straight down and back up once, without turning, to learn the arm's joint angles at the bottom;
  - c) winds the gripper back to the counterclockwise end;
  - d) down while turning, e) holding while turning, f) up while turning.

  It then prints for each part how long it took, how far it turned, how fast it went up or down, and how
  far it drifted sideways.
- **Cell 3**: turns the gripper back and brings the arm back to where it started.

**Before running:** gripper empty, nothing attached that could wind up, hand on the e-stop. Do not run anything
in another notebook that is connected to the arm at the same time. Run the cells in order.

In [ ]:
from robotic_testing.platforms import init_vla_platform

rt = init_vla_platform(exp_name='rotation_dip_demo', connect_db=False)
arm = rt.arm

In [ ]:
# ===== Demo 1/3: settings, then take the EMPTY gripper up into the air, pointing straight down =====
# This practises a dip with rotation in the air: the empty gripper goes down DIP_DEPTH_MM while turning,
# stays turning for HOLD_S, then comes back up while turning. Nothing touches anything.
# Before running: gripper EMPTY, nothing attached to it that could wind up, hand on the e-stop.
import time

ROTATION_SPEED_DEG_S = 10.0   # how fast the gripper turns clockwise (seen from above), degrees per second
DIP_SPEED_MM_S = 2.0          # going down ("into the ink"), mm per second
DIP_DEPTH_MM = 30.0           # how far it goes down while turning, mm
HOLD_S = 10.0                 # how long it stays at the bottom, still turning, seconds
LIFT_SPEED_MM_S = 1.0         # coming back up ("out of the ink"), mm per second

WIND_SPEED_DEG_S = 30.0       # speed for winding the gripper back before, and unwinding after
DEMO_SPOT = [347.9, 308.1, 500.0, 180.0, 0.0, 90.0]   # high above mid_station, gripper pointing straight down
J7_LIMIT = 350.0              # the last joint turns between -360 and +360 degrees; stay 10 degrees inside that

down_s = DIP_DEPTH_MM / DIP_SPEED_MM_S
up_s = DIP_DEPTH_MM / LIFT_SPEED_MM_S
total_turn = ROTATION_SPEED_DEG_S * (down_s + HOLD_S + up_s)
print(f'Plan: down {DIP_DEPTH_MM:g} mm in {down_s:.1f} s, hold {HOLD_S:g} s, up in {up_s:.1f} s, '
      f'turning {ROTATION_SPEED_DEG_S:g} deg/s all the time: {total_turn:.0f} degrees = {total_turn / 360:.2f} turns.')
if not (0 < DIP_SPEED_MM_S <= 50 and 0 < LIFT_SPEED_MM_S <= 50 and 0 < DIP_DEPTH_MM <= 80 and HOLD_S >= 0):
    raise SystemExit('A speed, the depth or the hold time is out of range. The arm did not move.')
if not 0 < ROTATION_SPEED_DEG_S <= 90:
    raise SystemExit('ROTATION_SPEED_DEG_S must be between 0 and 90. The arm did not move.')
if total_turn > 2 * J7_LIMIT:
    longest = 2 * J7_LIMIT / (down_s + HOLD_S + up_s)
    raise SystemExit(f'That is more than the {2 * J7_LIMIT:g} degrees the gripper can turn in one go. With these '
                     f'times, the rotation speed can be at most {longest:.1f} deg/s. The arm did not move.')


def at_speed(mm_s):
    """The arm's default motion settings with the speed swapped to mm_s."""
    return {**arm.config.pos_settings_dict['default'], 'speed': mm_s}


def j7():
    """Angle of the last joint (the one that turns the gripper), in degrees."""
    return arm.get_joint_angles()[6]


def turn_j7_to(angle, speed):
    """Turn only the last joint, to angle (degrees), at speed (degrees per second). Nothing else moves."""
    arm.set_servo_angle(servo_id=7, angle=angle, speed=speed, mvacc=200, wait=True)


start = arm.get_cartesian_pos()          # remembered, so cell 3 can bring the arm back here
print('Arm is now at:', [round(v, 1) for v in start])
pointing_down = abs(((start[3] - 180) + 180) % 360 - 180) <= 5 and abs(start[4]) <= 5
if not pointing_down or start[2] < 240:
    raise SystemExit('The gripper must point straight down and be at least 5 cm above the racks (z >= 240) '
                     'before this demo. The arm did not move.')
if input('Gripper empty, nothing attached that could wind up, hand on the e-stop? Type yes to go up into the air: '
         ).strip().lower() != 'yes':
    raise SystemExit('Cancelled, the arm did not move.')
arm.assert_ready()
print('-> straight up from here'); arm.move_to_pos([start[0], start[1], DEMO_SPOT[2], *start[3:]], **at_speed(30))
print('-> across, high above the bench, to above mid_station'); arm.move_to_pos(DEMO_SPOT, **at_speed(30))
j7_home = j7()                           # cell 3 turns the gripper back to this angle
print(f'In the air at {[round(v, 1) for v in arm.get_cartesian_pos()]}, last joint at {j7_home:.1f} degrees.')
print('Next: run cell 2.')

In [ ]:
# ===== Demo 2/3: down while turning, hold while turning, up while turning (empty gripper, in the air) =====
# a) find out which way the last joint turns for clockwise seen from above (a 10-degree turn and back);
# b) learn the arm's joint angles at the bottom: straight down DIP_DEPTH_MM without turning, and back up;
# c) wind the gripper back to the counterclockwise end, so the whole turn fits in the joint's range;
# d) down while turning, e) hold while turning, f) up while turning; then a short report of what happened.
here = arm.get_cartesian_pos()
if max(abs(a - b) for a, b in zip(here[:3], DEMO_SPOT[:3])) > 5:
    raise SystemExit('The arm is not up in the air at the demo spot. Run cell 1 first. Nothing moved.')
if input(f'Type yes to start ({total_turn:.0f} degrees of turning in all): ').strip().lower() != 'yes':
    raise SystemExit('Cancelled, the arm did not move.')
arm.assert_ready()
top_pose = arm.get_cartesian_pos()

print('a) which way is clockwise: a 10-degree clockwise turn (seen from above) and back')
before = j7()
arm.move_to_pos([*top_pose[:5], top_pose[5] - 10], **at_speed(10))   # yaw down = clockwise seen from above
change = j7() - before
arm.move_to_pos(top_pose, **at_speed(10))
if abs(abs(change) - 10) > 3:
    raise SystemExit(f'The last joint turned {change:+.1f} degrees for a 10-degree turn of the gripper; expected about '
                     f'10. Not safe to continue. Tell Claude.')
cw = 1 if change > 0 else -1
print('   clockwise seen from above = last joint angle going', 'UP' if cw == 1 else 'DOWN')

print(f'b) learning the joint angles: straight down {DIP_DEPTH_MM:g} mm and back up, without turning')
top = arm.get_joint_angles()
arm.move_to_pos([top_pose[0], top_pose[1], top_pose[2] - DIP_DEPTH_MM, *top_pose[3:]], **at_speed(10))
bottom = arm.get_joint_angles()
arm.move_to_pos(top_pose, **at_speed(10))
if abs(bottom[6] - top[6]) > 1:
    raise SystemExit('The last joint moved while going straight down; not safe to continue. Tell Claude.')


def move_joints_over(target, seconds, label):
    """Move all 7 joints together to target, taking about `seconds`; record the gripper's path meanwhile."""
    now = arm.get_joint_angles()
    biggest = max(abs(t - n) for t, n in zip(target, now))
    speed = biggest / seconds
    if speed > 180:
        raise SystemExit(f'{label}: that would need {speed:.0f} deg/s, more than the arm allows. Stopped.')
    p0, a0 = arm.get_cartesian_pos(), now[6]
    t0 = time.time()
    arm.set_servo_angle(angle=target, speed=speed, mvacc=200, wait=False)
    path = []
    while time.time() - t0 < seconds + 10:
        time.sleep(0.1)
        p = arm.get_cartesian_pos()
        path.append((time.time() - t0, p))
        if time.time() - t0 > 0.5 and not arm.get_is_moving():
            break
    t_end = path[-1][0] if path else time.time() - t0
    p1 = arm.get_cartesian_pos()
    drift = max([((p[0] - p0[0]) ** 2 + (p[1] - p0[1]) ** 2) ** 0.5 for _, p in path] or [0.0])
    print(f'   {label}: took {t_end:.1f} s (planned {seconds:.1f} s), turned {abs(j7() - a0):.0f} degrees, '
          f'went {p1[2] - p0[2]:+.1f} mm in height ({abs(p1[2] - p0[2]) / max(t_end, 1e-6):.2f} mm/s), '
          f'sideways drift at most {drift:.2f} mm')


ccw_end = -cw * J7_LIMIT
print(f'c) winding the gripper back counterclockwise, to {ccw_end:g} degrees')
turn_j7_to(ccw_end, WIND_SPEED_DEG_S)

print(f'd) down {DIP_DEPTH_MM:g} mm at {DIP_SPEED_MM_S:g} mm/s while turning {ROTATION_SPEED_DEG_S:g} deg/s')
move_joints_over(bottom[:6] + [j7() + cw * ROTATION_SPEED_DEG_S * down_s], down_s, 'down')
if HOLD_S > 0:
    print(f'e) hold {HOLD_S:g} s, still turning')
    t0 = time.time(); a0 = j7()
    turn_j7_to(a0 + cw * ROTATION_SPEED_DEG_S * HOLD_S, ROTATION_SPEED_DEG_S)
    print(f'   hold: took {time.time() - t0:.1f} s (planned {HOLD_S:g} s), turned {abs(j7() - a0):.0f} degrees')
print(f'f) up {DIP_DEPTH_MM:g} mm at {LIFT_SPEED_MM_S:g} mm/s while turning {ROTATION_SPEED_DEG_S:g} deg/s')
move_joints_over(top[:6] + [j7() + cw * ROTATION_SPEED_DEG_S * up_s], up_s, 'up')
print('Done. Send Claude a screenshot of this output, then run cell 3.')

In [ ]:
# ===== Demo 3/3: turn the gripper back, then bring the arm back to where it started =====
print(f'Turning the gripper back to {j7_home:.1f} degrees at {WIND_SPEED_DEG_S:g} deg/s...')
arm.assert_ready()
turn_j7_to(j7_home, WIND_SPEED_DEG_S)
arm.move_to_pos(DEMO_SPOT, **at_speed(20))
print('Going back: across, high above the bench, then down to the start position.')
arm.move_to_pos([start[0], start[1], DEMO_SPOT[2], *start[3:]], **at_speed(30))
arm.move_to_pos(start, **at_speed(20))
print('Back at:', [round(v, 1) for v in arm.get_cartesian_pos()], f'; last joint at {j7():.1f} degrees.')